# persona -> trl dpo with a lora adapter on smollm2-360m-instruct, then base vs lora answers on a style eval set

## imports

In [ ]:
from pathlib import Path
import json
import os
import random
import re
import shutil
import time

DATA_DIR = Path('./data')
OUTPUT_DIR = Path('./outputs/persona')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints' #lora adapters only, a few tens of mb each
for directory in [DATA_DIR, OUTPUT_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

#set before the hugging face imports so the model downloads into ./data, which is on the volume when run on modal
os.environ['HF_HOME'] = str((DATA_DIR / 'hf').resolve())

import pandas as pd
import matplotlib.pyplot as plt

import torch
import datasets
import peft
import transformers
import trl
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

## global env variables ->

In [ ]:
SEED = 42
BASE_MODEL = 'HuggingFaceTB/SmolLM2-360M-Instruct'
PERSONA_DATA = '../data/persona_lora/ramsay_style_preferences.json' #220 rows of prompt, chosen, rejected
EVAL_SET = '../eval_sets/persona_style_eval_40.json' #10 cooking, 10 coding, 10 general help, 10 safety sensitive
VAL_SIZE = 20 #held out pairs for the validation dpo loss and reward accuracy

#lora -> every linear layer of the llama style blocks, attention and mlp. a persona is a change in word choice and
#tone more than in what the model attends to, and the mlp layers hold most of that, so attention only would be too
#narrow. rank 16 is ~8.7m trainable parameters (2.4% of the model), plenty for 200 short pairs
LORA_RANK = 16
LORA_ALPHA = 32 #alpha / rank = 2, the usual scale, so the learning rate does not have to make up for a small scale
LORA_DROPOUT = 0.05 #light regularisation, the training set is tiny
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

#a short pilot grid, the selection rule is in the generation part below. lora needs a far higher rate than full
#dpo (1e-6 in the main runs) since only the small adapter moves, and 200 pairs give few updates
LEARNING_RATES = [1e-5, 2e-5, 5e-5, 1e-4]
#rpo adds alpha x the sft loss on the chosen answer to the dpo loss. plain dpo on 200 pairs can lower the loss by
#pushing both answers' likelihood down, and the higher rates then collapse into repeated insults. the sft term keeps
#the model able to actually write the chosen answers. 0 = plain dpo
RPO_ALPHAS = [0.0, 1.0]
BETA = 0.1 #the same strength as the from scratch runs
BATCH_SIZE = 8
GRAD_ACCUM = 1 #effective batch 8 -> 25 updates per epoch on 200 pairs, 75 in all
EPOCHS = 3 #one epoch is only 25 updates. the validation loss decides which step is kept, so a 3rd epoch is safe
WARMUP_RATIO = 0.1
MAX_LENGTH = 256 #chat template + prompt + answer, the longest training pair is ~110 tokens
MAX_PROMPT_LENGTH = 128
PRECISION = 'bf16' #'bf16' on modern cards, 'fp32' for a cpu smoke test
EVAL_EVERY = 5 #optimizer steps between validation passes and adapter saves
MAX_STEPS = -1 #-1 = the full epochs. a small number gives a quick end to end check

#decoding is identical for the base and the lora model, so a judged difference comes from the adapter
MAX_NEW_TOKENS = 384
REPETITION_PENALTY = 1.0 #1.0 = off, greedy like the main comparison
GENERATE_NAME = 'generate' #a decoding ablation writes its answers, selection and pair files to its own folder
GEN_BATCH_SIZE = 16
MAX_PROMPTS = 0 #0 = the whole eval set
MIN_STOP_RATE = 0.9 #an adapter is only kept if at least this share of its validation answers end by themselves

In [ ]:
GENERATE_DIR = OUTPUT_DIR / GENERATE_NAME
GENERATE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'using device {device}')
if torch.cuda.is_available():
    print(f'gpu {torch.cuda.get_device_name(0)}  '
          f'memory {torch.cuda.get_device_properties(0).total_memory / 1024 ** 3:.0f} gb')
print(f'torch {torch.__version__}  transformers {transformers.__version__}  trl {trl.__version__}  '
      f'peft {peft.__version__}  datasets {datasets.__version__}')

## data -> trl's conversational format, a seeded train / validation split

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'eos {tokenizer.eos_token!r}  pad {tokenizer.pad_token!r}')

rows = json.loads(Path(PERSONA_DATA).read_text())
random.Random(SEED).shuffle(rows) #its own generator, the split does not depend on anything else drawn above
train_rows, val_rows = rows[VAL_SIZE:], rows[:VAL_SIZE]


def to_conversation(row):
    #trl applies the model's chat template itself when the fields are message lists. the prompt is the user turn,
    #chosen and rejected are the two candidate assistant turns that follow it
    return {
        'prompt': [{'role': 'user', 'content': row['prompt']}],
        'chosen': [{'role': 'assistant', 'content': row['chosen']}],
        'rejected': [{'role': 'assistant', 'content': row['rejected']}],
    }


train_dataset = Dataset.from_list([to_conversation(row) for row in train_rows])
val_dataset = Dataset.from_list([to_conversation(row) for row in val_rows])
(OUTPUT_DIR / 'split.json').write_text(json.dumps({'train': [row['prompt'] for row in train_rows],
                                                    'val': [row['prompt'] for row in val_rows]}, indent=2))
print(f'{len(train_dataset)} train / {len(val_dataset)} val pairs')

#the exact text the model is trained on, checked by eye before any gpu time is spent
example = train_dataset[0]
print(tokenizer.apply_chat_template(example['prompt'] + example['chosen'], tokenize=False))

## lora dpo -> one run per learning rate, the reference is the base model with the adapter switched off

In [ ]:
lora_config = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                         target_modules=LORA_TARGET_MODULES, task_type='CAUSAL_LM')


def train_persona(learning_rate, rpo_alpha):
    run_name = f'persona_lr{learning_rate:g}' + (f'_rpo{rpo_alpha:g}' if rpo_alpha else '')
    adapter_dir = CHECKPOINT_DIR / run_name
    history_path = OUTPUT_DIR / f'{run_name}_history.csv'
    done_path = OUTPUT_DIR / f'{run_name}_done.json'
    if done_path.exists(): #a finished run is not repeated when the notebook is restarted
        print(f'{run_name} already finished, reusing {adapter_dir}')
        return json.loads(done_path.read_text())

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32) #fp32 weights, bf16 autocast

    args = DPOConfig(
        output_dir=str(OUTPUT_DIR / 'trainer' / run_name), #intermediate adapter saves, only the best is kept below
        beta=BETA,
        rpo_alpha=rpo_alpha or None, #none = plain dpo
        learning_rate=learning_rate,
        lr_scheduler_type='cosine',
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.0, #as in the from scratch dpo runs
        max_grad_norm=1.0,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=VAL_SIZE, #the whole validation split in one pass
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        max_length=MAX_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        bf16=PRECISION == 'bf16',
        gradient_checkpointing=False, #a 360m model at 256 tokens fits easily, checkpointing would only slow it down
        eval_strategy='steps',
        eval_steps=EVAL_EVERY,
        logging_steps=EVAL_EVERY,
        save_strategy='steps',
        save_steps=EVAL_EVERY,
        save_total_limit=2, #the best and the latest adapter
        load_best_model_at_end=True, #the adapter that is saved below is the one with the lowest validation loss
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        seed=SEED,
    )
    #ref_model is left out on purpose -> with a peft config trl uses the same base weights with the adapter
    #disabled as the frozen reference, so no second copy of the model is loaded
    trainer = DPOTrainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=val_dataset,
                         processing_class=tokenizer, peft_config=lora_config)
    trainer.model.print_trainable_parameters() #the adapter is the only trainable part

    start_metrics = trainer.evaluate() #step 0 -> adapter still zero, so the policy equals the reference
    run_start = time.time()
    trainer.train()

    #trl's log history mixes training rows ('loss') and validation rows ('eval_loss'), one row per logging event
    history = pd.DataFrame([{**start_metrics, 'step': 0}] + trainer.state.log_history)
    history.to_csv(history_path, index=False)
    trainer.save_model(str(adapter_dir)) #the best adapter, load_best_model_at_end swapped it in
    tokenizer.save_pretrained(adapter_dir)
    shutil.rmtree(args.output_dir) #the trainer's intermediate saves carry optimizer state, the best adapter is kept above

    val_rows_only = history.dropna(subset=['eval_loss'])
    best = val_rows_only.loc[val_rows_only['eval_loss'].idxmin()]
    summary = {
        'run': run_name, 'learning_rate': learning_rate, 'rpo_alpha': rpo_alpha, 'steps': int(trainer.state.global_step),
        'best_step': int(best['step']), 'best_eval_loss': float(best['eval_loss']),
        'best_eval_reward_accuracy': float(best['eval_rewards/accuracies']),
        'best_eval_reward_margin': float(best['eval_rewards/margins']),
        'final_eval_loss': float(val_rows_only.iloc[-1]['eval_loss']),
        'trainable_parameters': sum(p.numel() for p in trainer.model.parameters() if p.requires_grad),
        'seconds': time.time() - run_start,
    }
    done_path.write_text(json.dumps(summary, indent=2))
    print(f"{run_name}: best eval loss {summary['best_eval_loss']:.4f} at step {summary['best_step']}, "
          f"reward accuracy {summary['best_eval_reward_accuracy']:.2f}, margin {summary['best_eval_reward_margin']:.3f}")

    del trainer, model
    torch.cuda.empty_cache()
    return summary


pilot = pd.DataFrame([train_persona(learning_rate, rpo_alpha) for rpo_alpha in RPO_ALPHAS
                      for learning_rate in LEARNING_RATES])
pilot['rpo_alpha'] = pilot['rpo_alpha'].fillna(0.0) if 'rpo_alpha' in pilot else 0.0 #runs saved before rpo existed
pilot.to_csv(OUTPUT_DIR / 'persona_pilot.csv', index=False)
print(pilot.round(4).to_string(index=False))

In [ ]:
#loss, reward accuracy and reward margin for every run, one row per rpo setting, validation as markers over the
#training curve. with rpo the loss also contains the sft term, so its scale differs between the rows
figure, axes = plt.subplots(len(RPO_ALPHAS), 3, figsize=(15, 4 * len(RPO_ALPHAS)), squeeze=False)
for row_axes, rpo_alpha in zip(axes, RPO_ALPHAS):
    for run_name, learning_rate in pilot[pilot['rpo_alpha'] == rpo_alpha][['run', 'learning_rate']].values:
        history = pd.read_csv(OUTPUT_DIR / f'{run_name}_history.csv')
        train_rows_h = history.dropna(subset=['loss'])
        val_rows_h = history.dropna(subset=['eval_loss'])
        for axis, metric in zip(row_axes, ['loss', 'rewards/accuracies', 'rewards/margins']):
            line, = axis.plot(train_rows_h['step'], train_rows_h[metric], alpha=0.5,
                              label=f'lr {learning_rate:g} train (mean of {EVAL_EVERY} steps)')
            axis.plot(val_rows_h['step'], val_rows_h[f'eval_{metric}'], marker='o', color=line.get_color(),
                      label=f'lr {learning_rate:g} validation')
    setting = f'rpo alpha {rpo_alpha:g}' if rpo_alpha else 'plain dpo'
    for axis, label in zip(row_axes, ['loss', 'reward accuracy', 'reward margin']):
        axis.set_xlabel('optimizer step')
        axis.set_ylabel(label)
        axis.set_title(f'persona lora, {setting}: {label}')
    row_axes[0].legend(fontsize=7)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'persona_training.png', dpi=150)
plt.show()

## generation helpers -> the same chat template and greedy decoding for every model

In [ ]:
generation_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
generation_tokenizer.padding_side = 'left' #a decoder only model continues from the last token of each row
END_OF_TURN_ID = generation_tokenizer.convert_tokens_to_ids('<|im_end|>') #what every chat answer ends with


@torch.no_grad()
def generate_all(model, instructions):
    outputs, finished = [], []
    for start in range(0, len(instructions), GEN_BATCH_SIZE):
        #the same chat template the training pairs went through, the default system turn included
        prompts = [generation_tokenizer.apply_chat_template([{'role': 'user', 'content': instruction}],
                                                            tokenize=False, add_generation_prompt=True)
                   for instruction in instructions[start:start + GEN_BATCH_SIZE]]
        encoded = generation_tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(device)
        output_ids = model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                    repetition_penalty=REPETITION_PENALTY, eos_token_id=END_OF_TURN_ID,
                                    pad_token_id=generation_tokenizer.pad_token_id)
        new_ids = output_ids[:, encoded['input_ids'].shape[1]:]
        outputs += [text.strip() for text in generation_tokenizer.batch_decode(new_ids, skip_special_tokens=True)]
        finished += (new_ids == END_OF_TURN_ID).any(dim=1).tolist() #false = ran into MAX_NEW_TOKENS
    return outputs, finished


def load_generator(adapter_dir=None):
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    if adapter_dir is not None:
        #folded into the base weights, so the lora model generates exactly like a plain model of the same size
        model = PeftModel.from_pretrained(model, adapter_dir).merge_and_unload()
    return model.to(device).eval()


#a rough automatic style check next to the judge -> exclamation marks and the dataset's signature insults
INSULT_PATTERN = re.compile(r'\byou (absolute |complete |reckless |utter )?(donkey|donut|muppet|amateur|maniac|'
                            r'disaster|idiot|clown|lunatic|menace|sandwich)', flags=re.IGNORECASE)


def style_stats(outputs, finished):
    return {'reached_eos': sum(finished) / len(finished),
            'mean_tokens': sum(len(generation_tokenizer.encode(output)) for output in outputs) / len(outputs),
            'exclamations_per_answer': sum(output.count('!') for output in outputs) / len(outputs),
            'insult_rate': sum(bool(INSULT_PATTERN.search(output)) for output in outputs) / len(outputs)}


GENERATORS = {'smollm2_base': None, **{run: CHECKPOINT_DIR / run for run in pilot['run']}} #name -> adapter or none

## adapter selection -> the strongest preference that still ends its answers, checked on held out validation prompts

In [ ]:
#the validation dpo loss only says how confidently the adapter ranks the ramsay answer above the neutral one. it kept
#falling with every higher rate while the answers of the highest rate collapsed into repeated insults, so it cannot
#pick the adapter on its own. instead every adapter answers the 20 validation prompts (held out from training and not
#part of the eval set). among the adapters whose answers still end by themselves often enough, the one with the
#largest validation reward margin, the strongest preference for the persona answer, is kept
validation_prompts = [row['prompt'] for row in val_rows]
coherence = []
for name, adapter_dir in GENERATORS.items():
    model = load_generator(adapter_dir)
    outputs, finished = generate_all(model, validation_prompts)
    coherence.append({'run': name, **{f'val_{key}': value for key, value in style_stats(outputs, finished).items()}})
    print(f"\n[{name} | validation prompt]\n>>> {validation_prompts[0]}\n{outputs[0][:300]}")
    del model
    torch.cuda.empty_cache()

pilot = pilot.merge(pd.DataFrame(coherence), on='run')
pilot.to_csv(GENERATE_DIR / 'selection.csv', index=False) #the selection depends on the decoding, so it sits with it
print(pilot[['run', 'learning_rate', 'rpo_alpha', 'best_eval_reward_margin', 'val_reached_eos', 'val_mean_tokens',
             'val_exclamations_per_answer']].round(3).to_string(index=False))

coherent = pilot[pilot['val_reached_eos'] >= MIN_STOP_RATE]
#fallback -> the gentlest run, if even that one rambles the selection says so in the printout below
selected = (coherent.loc[coherent['best_eval_reward_margin'].idxmax()] if len(coherent)
            else pilot.loc[pilot['best_eval_reward_margin'].idxmin()])
print(f"selected {selected['run']} (margin {selected['best_eval_reward_margin']:.2f}): {selected['val_reached_eos']:.0%} of validation "
      f"answers end by themselves, {selected['val_exclamations_per_answer']:.1f} exclamation marks per answer")

## generation on the persona eval set -> the base model and every adapter, same greedy decoding

In [ ]:
eval_rows = json.loads(Path(EVAL_SET).read_text())
eval_rows = eval_rows[:MAX_PROMPTS] if MAX_PROMPTS else eval_rows
instructions = [row['instruction'] for row in eval_rows]

summary_rows = []
for name, adapter_dir in GENERATORS.items():
    model = load_generator(adapter_dir)
    outputs, finished = generate_all(model, instructions)
    entries = [{'id': row['id'], 'instruction': row['instruction'], 'output': output, 'generator': name,
                'dataset': 'persona_eval', 'category': row['category']} for row, output in zip(eval_rows, outputs)]
    (GENERATE_DIR / f'{name}__persona_eval.json').write_text(json.dumps(entries, indent=2, ensure_ascii=False))

    #per category and over all prompts -> the safety prompts are where an insulting persona would do harm
    for category in ['all'] + sorted({row['category'] for row in eval_rows}):
        part = [index for index, row in enumerate(eval_rows) if category in ('all', row['category'])]
        summary_rows.append({'generator': name, 'selected': name == selected['run'], 'category': category,
                             'prompts': len(part),
                             **style_stats([outputs[i] for i in part], [finished[i] for i in part])})
    for entry in entries[:1] + entries[30:31]: #one cooking and one safety answer of each model, read by eye
        print(f"\n[{name} | {entry['category']}]\n>>> {entry['instruction']}\n{entry['output'][:400]}")
    del model
    torch.cuda.empty_cache()

generation_summary = pd.DataFrame(summary_rows)
generation_summary.to_csv(GENERATE_DIR / 'generation_summary.csv', index=False)
print(generation_summary.round(3).to_string(index=False))

## pair files -> every adapter vs the base model on the same prompts, a/b order randomised, identity kept hidden

In [ ]:
#one file per run, so the judge gives a style / quality curve across rates and rpo, not only the selected point
base_entries = json.loads((GENERATE_DIR / 'smollm2_base__persona_eval.json').read_text())
for name in pilot['run']:
    lora_entries = json.loads((GENERATE_DIR / f'{name}__persona_eval.json').read_text())
    order_rng = random.Random(f'{SEED}-{name}-persona_eval') #reproducible a/b coin flips, one per prompt
    pairs = []
    for base_entry, lora_entry in zip(base_entries, lora_entries):
        first, second = (lora_entry, base_entry) if order_rng.random() < 0.5 else (base_entry, lora_entry)
        pairs.append({
            'pair_id': f"{name}_vs_smollm2_base__{base_entry['id']}", 'id': base_entry['id'],
            'dataset': 'persona_eval', 'instruction': base_entry['instruction'],
            'response_a': first['output'], 'response_b': second['output'], #only these and the instruction are judged
            'model_a': first['generator'], 'model_b': second['generator'], 'category': base_entry['category'],
        })
    pairs_path = GENERATE_DIR / f'{name}_vs_smollm2_base__persona_eval_pairs.jsonl'
    pairs_path.write_text(''.join(json.dumps(pair, ensure_ascii=False) + '\n' for pair in pairs))
    print(f"{pairs_path.name}: {len(pairs)} pairs, lora shown as a in {sum(p['model_a'] == name for p in pairs)}")

#the configuration actually used, for the report's table
run_config = {
    'base_model': BASE_MODEL, 'seed': SEED, 'train_size': len(train_dataset), 'val_size': len(val_dataset),
    'lora': {'rank': LORA_RANK, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'target_modules': LORA_TARGET_MODULES},
    'dpo': {'trainer': f'trl {trl.__version__} DPOTrainer', 'beta': BETA, 'reference': 'base model, adapter disabled',
            'learning_rates_tried': LEARNING_RATES, 'rpo_alphas_tried': RPO_ALPHAS, 'selected_run': selected['run'],
            'selected_learning_rate': float(selected['learning_rate']), 'selected_rpo_alpha': float(selected['rpo_alpha']),
            'checkpoint_within_run': 'lowest validation dpo loss',
            'selection_across_runs': f'largest validation reward margin among runs whose validation answers end by '
                                     f'themselves >= {MIN_STOP_RATE:.0%}',
            'optimizer': 'adamw', 'schedule': 'cosine', 'warmup_ratio': WARMUP_RATIO, 'weight_decay': 0.0,
            'max_grad_norm': 1.0, 'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM, 'epochs': EPOCHS,
            'max_length': MAX_LENGTH, 'precision': PRECISION},
    'generation': {'decoding': 'greedy', 'max_new_tokens': MAX_NEW_TOKENS, 'repetition_penalty': REPETITION_PENALTY,
                   'chat_template': 'smollm2 default, with its default system turn'},
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}
(GENERATE_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2))